## Purpose
The purpose of this script is to extract groundwater pumping data from the USGS

## Install libraries

In [30]:
# !pip install pynhd
from pynhd import WaterData
import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
# requests provides better error handling
# when retrieving API requests
import requests
# urlencode ensures query parameters
# are correctly URL-encoded
from urllib.parse import urlencode

In [32]:
# Specify your desired CSV files
BASE_URL = 'https://water.usgs.gov/nwaa-data/data-file-directory?path=data/'
csv_folders = ['water-use/wu-irrigation-wd/irrwdgw/', #Ground water used for crop irrigation
               'water-use/wu-irrigation-wd/irrwdsw/'] #Surface water used for irrigation
csv_files = ['irrwdgw_wu-irrigation-wd_CONUS_200001-202012_wide.csv',
             'irrwdsw_wu-irrigation-wd_CONUS_200001-202012_wide.csv']
for csv_folder, csv_file in zip(csv_folders, csv_files):
    print(f'File URL: {BASE_URL + csv_folder + csv_file}')
    print(f'CSV file to be saved: {csv_file}')

# Specify state
state_id = 'AZ'

File URL: https://water.usgs.gov/nwaa-data/data-file-directory?path=data/water-use/wu-irrigation-wd/irrwdgw/irrwdgw_wu-irrigation-wd_CONUS_200001-202012_wide.csv
CSV file to be saved: irrwdgw_wu-irrigation-wd_CONUS_200001-202012_wide.csv
File URL: https://water.usgs.gov/nwaa-data/data-file-directory?path=data/water-use/wu-irrigation-wd/irrwdsw/irrwdsw_wu-irrigation-wd_CONUS_200001-202012_wide.csv
CSV file to be saved: irrwdsw_wu-irrigation-wd_CONUS_200001-202012_wide.csv


In [33]:
for csv_folder, csv_file in zip(csv_folders, csv_files):
    try:
        # Send a GET request to download the file
        response = requests.get(BASE_URL + csv_folder + csv_file,
                                timeout=10)
    
        # Raise an exception if the status code indicates an error
        response.raise_for_status()
    
        # Save the content to a file with the same name
        with open(csv_file, 'wb') as file:
            file.write(response.content)
        print(f"File '{csv_file}' downloaded and saved successfully.")
    
    except requests.exceptions.HTTPError as http_err:
        print(f"HTTP error occurred: {http_err}")
    except requests.exceptions.ConnectionError:
        print("A connection error occurred. Please check your internet connection and the URL.")
    except requests.exceptions.Timeout:
        print("The request timed out. Try increasing the timeout or check your network.")
    except requests.exceptions.RequestException as req_err:
        print(f"An error occurred while handling the request: {req_err}")
    except IOError as io_err:
        print(f"An I/O error occurred while saving the file: {io_err}")


File 'irrwdgw_wu-irrigation-wd_CONUS_200001-202012_wide.csv' downloaded and saved successfully.
File 'irrwdsw_wu-irrigation-wd_CONUS_200001-202012_wide.csv' downloaded and saved successfully.


## Get all HUC12 IDs completely within a state (Arizona)

In [11]:
# Define base USGS GeoServer API base URL
GS_BASE_URL = 'https://api.water.usgs.gov/geoserver/wmadata/ows'

# Define GeoServer API call parameters
# to return all HUC12 IDs in the specified state
params = {
    'service': 'WFS',
    'request': 'GetFeature',
    'version': '1.0.0',
    'typename': 'feature:wbd12_20201006',
    'propertyname': 'huc12',
    'outputformat': 'csv',
    #'cql_filter': f'states like \'%{state_id}%\''  # Get all HUC12s overlapping state_id
    'cql_filter': f'states like \'{state_id}\''  # Get all HUC12s completely within state_id
}

# Create the GeoServer API URL using urlencode for proper URL encoding
gs_api_url = f'{GS_BASE_URL}?{urlencode(params)}'

# Create the CSV file name using f-string
huc12s_csv_file = 'wbd12_20201006.csv'

print(f'GeoServer API URL: {gs_api_url} \nCSV file to be saved: {huc12s_csv_file}')

GeoServer API URL: https://api.water.usgs.gov/geoserver/wmadata/ows?service=WFS&request=GetFeature&version=1.0.0&typename=feature%3Awbd12_20201006&propertyname=huc12&outputformat=csv&cql_filter=states+like+%27AZ%27 
CSV file to be saved: wbd12_20201006.csv


In [12]:
# Use the requests package for downloading the HUC12 IDs
try:
    response = requests.get(gs_api_url, stream=True)
    response.raise_for_status()  # Raise an error for bad status codes
    with open(huc12s_csv_file, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f'File downloaded successfully: {huc12s_csv_file}')
except requests.exceptions.HTTPError as http_err:
    print(f'HTTP error occurred: {http_err}')
except Exception as err:
    print(f'An error occurred: {err}')

File downloaded successfully: wbd12_20201006.csv


In [15]:
# Read the HUC12s CSV file into a dataframe
state_huc12s = pd.read_csv(
    huc12s_csv_file,
    dtype={'huc12': str}  # Load huc12 ids as strings
)

## Step 3) Filter the CONUS-wide data down to only the HUC12s completely within a state (Arizona)

In [16]:
dfgw = pd.read_csv(csv_files[0])
# To ensure the year_month column is kept,
# set that column as the dataframe index
dfgw.set_index('year_month', inplace=True)
# Only keep the columns corresponding to
# HUC12 IDs within the specified state
dfgw = dfgw[dfgw.columns[dfgw.columns.isin(
    state_huc12s['huc12'])]]

In [17]:
dfsw = pd.read_csv(csv_files[1])
# To ensure the year_month column is kept,
# set that column as the dataframe index
dfsw.set_index('year_month', inplace=True)
# Only keep the columns corresponding to
# HUC12 IDs within the specified state
dfsw = dfsw[dfsw.columns[dfsw.columns.isin(
    state_huc12s['huc12'])]]

In [22]:
import requests
from urllib.parse import urlencode

GS_BASE_URL = 'https://api.water.usgs.gov/geoserver/wmadata/ows'

params = {
    'service': 'WFS',
    'request': 'GetFeature',
    'version': '1.0.0',
    'typename': 'wmadata:huc12',
    'propertyname': 'huc12,name,states,tohuc,areaacres',
    'outputformat': 'csv',
    'cql_filter': "name like '%Gila Bend%'"
}

url = f'{GS_BASE_URL}?{urlencode(params)}'
r = requests.get(url, timeout=30)
print(r.text)

FID,huc12,tohuc,areaacres,name,states
huc12.fid-7ef05f9f_19efa7c5c31_1d67,150701010408,150701010409,14955.458639,Gila Bend Municipal Airport Area,AZ
huc12.fid-7ef05f9f_19efa7c5c31_1d68,150701010705,150701010709,8513.788252,Gila Bend Mountains,AZ
huc12.fid-7ef05f9f_19efa7c5c31_1d69,150701010408,150701010409,14955.458639,Gila Bend Municipal Airport Area,AZ
huc12.fid-7ef05f9f_19efa7c5c31_1d6a,150701010705,150701010709,8513.788252,Gila Bend Mountains,AZ
huc12.fid-7ef05f9f_19efa7c5c31_1d6b,150701010408,150701010409,14955.458639,Gila Bend Municipal Airport Area,AZ
huc12.fid-7ef05f9f_19efa7c5c31_1d6c,150701010705,150701010709,8513.788252,Gila Bend Mountains,AZ



In [26]:
print(r.status_code)
print(r.headers.get('content-type'))
print(r.text[:500])

400
application/json; charset=utf-8
{"message":"Unsupported file format. Supported file formats are csv, shapefile, geojson, kml","error":"Bad Request","statusCode":400}


In [29]:
# Drop the bogus outSR param, just ask for shapefile format directly
url = "https://gisdata2016-11-18t150447874z-azwater.opendata.arcgis.com/datasets/azwater::groundwater-basin.zip"
r = requests.get(url, timeout=60)
print(r.status_code, r.headers.get('content-type'))
print(r.text[:300] if r.status_code != 200 else "looks ok")

400 application/json; charset=utf-8
{"message":"Unsupported file format. Supported file formats are csv, shapefile, geojson, kml","error":"Bad Request","statusCode":400}
